In [1]:
import tifffile
import numpy as np
import os
from tqdm import tqdm

# --- CONFIGURAÇÃO ---
input_path = r"D:\User data\InesMarques\20250920\train_hindbrain\data\preprocradius100cubic\fish2may_halffish-Airyscan Processing_airyscandownsampledhindbrain_onlynuclei_preproc.tif"
output_base_folder = r"d:\User data\InesMarques\20250920\train_hindbrain\data\train_crops\preprocessimage100radiuscubic"
crop_size = 516
n_crops = 20
# --- FIM DA CONFIGURAÇÃO ---

# Criar a pasta base de saída se não existir
os.makedirs(output_base_folder, exist_ok=True)

print(f"A carregar o stack: {os.path.basename(input_path)}")
stack = tifffile.imread(input_path)

# Assumindo a ordem de eixos (Z, Y, X)
Z, Y, X = stack.shape
print(f"Stack carregado com sucesso. Dimensões (Z, Y, X): ({Z}, {Y}, {X})")

# Função para gerar fatias 2D aleatórias
def generate_random_crops(stack, output_folder, crop_size, n_crops, axis):
    os.makedirs(output_folder, exist_ok=True)
    for i in tqdm(range(n_crops), desc=f"Gerando crops {axis}-slice"):
        if axis == 'XY':
            z_idx = np.random.randint(0, Z)
            y_start = np.random.randint(0, Y - crop_size + 1) if Y >= crop_size else 0
            x_start = np.random.randint(0, X - crop_size + 1) if X >= crop_size else 0
            crop = stack[z_idx, y_start:y_start + crop_size, x_start:x_start + crop_size]
        elif axis == 'XZ':
            y_idx = np.random.randint(0, Y)
            z_start = np.random.randint(0, Z - crop_size + 1) if Z >= crop_size else 0
            x_start = np.random.randint(0, X - crop_size + 1) if X >= crop_size else 0
            crop = stack[z_start:z_start + crop_size, y_idx, x_start:x_start + crop_size]
        elif axis == 'YZ':
            x_idx = np.random.randint(0, X)
            z_start = np.random.randint(0, Z - crop_size + 1) if Z >= crop_size else 0
            y_start = np.random.randint(0, Y - crop_size + 1) if Y >= crop_size else 0
            crop = stack[z_start:z_start + crop_size, y_start:y_start + crop_size, x_idx]
        else:
            raise ValueError("Eixo inválido. Use 'XY', 'XZ' ou 'YZ'.")
        
        # Salvar o crop
        crop_filename = f"crop_{i:03d}.tif"
        tifffile.imwrite(os.path.join(output_folder, crop_filename), crop)

# Gerar e guardar os crops para cada orientação
generate_random_crops(stack, os.path.join(output_base_folder, 'xy_slices'), crop_size, n_crops, 'XY')
generate_random_crops(stack, os.path.join(output_base_folder, 'xz_slices'), crop_size, n_crops, 'XZ')
generate_random_crops(stack, os.path.join(output_base_folder, 'yz_slices'), crop_size, n_crops, 'YZ')

print("\n--- Processo Concluído! ---")
print(f"Todos os crops foram guardados em: {output_base_folder}")




A carregar o stack: fish2may_halffish-Airyscan Processing_airyscandownsampledhindbrain_onlynuclei_preproc.tif
Stack carregado com sucesso. Dimensões (Z, Y, X): (842, 1805, 905)


Gerando crops YZ-slice: 100%|██████████| 20/20 [00:00<00:00, 163.86it/s]


--- Processo Concluído! ---
Todos os crops foram guardados em: d:\User data\InesMarques\20250920\train_hindbrain\data\train_crops\preprocessimage100radiuscubic


mesmo de cima mas com contagem do tempo

In [ ]:
import numpy as np
import tifffile
import torch
import os
import time
from cellpose import models, io

# -------- CONFIGURAÇÃO --------
input_file = "outcrossdsred_fish5_stackgood-slice1to100_3channelsjointdeconvolutionstichedbacksubtractionmedianfilter50slices8bit.tif"
output_dir = "./output_masks"
model_path = "models/CP_20250908_175220"  # seu modelo custom 2D

diameter = 57
stitch_threshold = 0.4
flow_thresh = 0.7
cellprob_thresh = -2.5
min_size = 500
channel_to_segment = 0
anisotropy = 7.6

# número de slices de teste
num_test_slices = 50
# -------------------------------

def log_time(start, msg):
    end = time.perf_counter()
    print(f"⏱ {msg}: {end - start:.2f} s")
    return end

# início geral
t0 = time.perf_counter()

# checar GPU
use_gpu = torch.cuda.is_available()
print("🚀 GPU disponível:", use_gpu)
t1 = log_time(t0, "Check GPU")

# carregar stack completo
stack_full = tifffile.imread(input_file)
print("Stack shape:", stack_full.shape)
t2 = log_time(t1, "Carregamento do stack completo")

# pegar apenas os primeiros N slices
stack = stack_full[:num_test_slices]
print("Stack para teste shape:", stack.shape)
t3 = log_time(t2, f"Seleção dos primeiros {num_test_slices} slices")

# criar pasta de saída
os.makedirs(output_dir, exist_ok=True)
t4 = log_time(t3, "Criação/checagem da pasta de saída")

# carregar modelo custom
model = models.CellposeModel(gpu=use_gpu, pretrained_model=model_path)
t5 = log_time(t4, "Carregamento do modelo Cellpose")

# definir canais
channels = [channel_to_segment, channel_to_segment]

# segmentação slice-wise com stitching
masks, flows, styles = model.eval(
    stack,
    channels=channels,
    diameter=diameter,
    do_3D=False,
    stitch_threshold=stitch_threshold,
    flow_threshold=flow_thresh,
    cellprob_threshold=cellprob_thresh,
    min_size=min_size,
    z_axis=0,
    anisotropy=anisotropy,
    progress=True
)
t6 = log_time(t5, "Segmentação Cellpose")

# salvar máscaras 3D usando prefixo
mask_prefix = os.path.join(output_dir, "mask_test_slices")
io.masks_flows_to_seg(stack, masks, flows, mask_prefix, diams=diameter)
t7 = log_time(t6, "Salvar máscaras e fluxos")

# total
log_time(t0, "Tempo total")
